<img src="logo.png" alt="Vegeta" width="240">

# Designing a twin-motor fixed-wing drone — wing structure, whole-aircraft aerodynamics, printed nacelles

A 1 m-span electric fixed wing with **two motors in nacelles on the wing** (mirror symmetric about the
fuselage). The notebook goes: mission and mass budget → parametric aircraft (Dedalus, NACA 4-digit wing
with taper and dihedral) → wing structure for a pull-up and an engine-out case (Talos) → a thicker-wing
revision compared in a `vegeta.core` workspace → RANS of the whole aircraft at 4° (Aeromant) giving lift,
drag, level-flight speed and an endurance estimate → nacelle slicing (Mellonia).

Every load, material and efficiency below is an **explicit input written here**; the numbers are
illustrative and coarse (fast meshes), meant to be replaced with yours. Nothing runs unless you run it.

In [ ]:
import math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import dedalus, talos, aeromant, mellonia, core
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
from tqdm.auto import tqdm

RUNS = Path("_runs/fixed_wing"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
RHO, G = 1.2, 9.81                      # air density [kg/m^3], gravity [m/s^2]

## 1. Mission and mass budget

Mapping/patrol drone: 1 m span, cruise around 14 m/s, 3S 5000 mAh battery, 150 g payload. Twin motors
give redundancy (engine-out is a structural case below) and keep the nose free for the payload.
The airframe mass is estimated from the CAD **surface area** — the wing, fuselage and tail are printed
as thin shells — with an explicit areal density; the nacelles are solid-ish and use volume.

In [ ]:
parts = pd.DataFrame([
    ("motors 2212 (2x)",          2 * 55.0),
    ("propellers 9x6 (2x)",       2 * 12.0),
    ("ESC 30 A (2x)",             2 * 25.0),
    ("battery 3S 5000 mAh",       380.0),
    ("flight controller + GPS",   35.0),
    ("servos (4x)",               4 * 12.0),
    ("receiver, wiring, bolts",   45.0),
    ("payload (camera)",          150.0),
], columns=["part", "mass_g"]).set_index("part")

SHELL_AREAL_DENSITY_G_MM2 = 1.1e-4     # 1.1 kg/m^2: ~1.4 mm LW-PLA shell at 0.8 g/cm^3 (assumption)
NACELLE_DENSITY_G_MM3 = 0.5e-3         # PLA at ~40 % effective density (walls + infill), assumption
THRUST_PER_MOTOR_N = 8.0               # static thrust of a 2212 / 9x6 on 3S (datasheet-style value)
BATTERY_WH, USABLE_FRACTION, PROP_EFFICIENCY = 11.1 * 5.0, 0.8, 0.55
parts

## 2. Parametric aircraft (Dedalus)

`FixedWing` builds a NACA `camber/camber_pos/thickness` wing as a constant-chord centre section (inside
the fuselage, with its own faces so it can be clamped in the FEA) plus two tapered outer panels with
dihedral, a revolved fuselage, flat-plate tail surfaces and the two nacelles — built once and mirrored,
so the aircraft is symmetric by construction. `part` chooses `aircraft`, `wing` (wing + nacelles, for the
structure) or `nacelle` (for printing). `angle_of_attack_deg` rotates the whole aircraft for the CFD.

In [ ]:
design_file = RUNS / "wing.py"
design_file.write_text('''import math
import cadquery as cq
from vegeta.dedalus import Design, Parameter


def naca4(camber, camber_pos, thickness, n=40, te_cut=0.97):
    """NACA 4-digit section as (upper, lower) point lists from the trailing edge to the leading edge,
    chord 1, truncated at ``te_cut`` so the trailing edge has a small finite thickness."""
    m, p, t = camber, camber_pos, thickness
    xs = [te_cut * 0.5 * (1 - math.cos(math.pi * i / n)) for i in range(n + 1)]   # cosine spacing, LE dense
    upper, lower = [], []
    for x in xs:
        yt = 5 * t * (0.2969 * math.sqrt(x) - 0.1260 * x - 0.3516 * x**2 + 0.2843 * x**3 - 0.1015 * x**4)
        if p > 0 and x < p:
            yc, dyc = m / p**2 * (2 * p * x - x**2), 2 * m / p**2 * (p - x)
        elif p > 0:
            yc, dyc = m / (1 - p)**2 * (1 - 2 * p + 2 * p * x - x**2), 2 * m / (1 - p)**2 * (p - x)
        else:
            yc, dyc = 0.0, 0.0
        th = math.atan(dyc)
        upper.append((x - yt * math.sin(th), yc + yt * math.cos(th)))
        lower.append((x + yt * math.sin(th), yc - yt * math.cos(th)))
    return upper[::-1], lower[::-1]          # both start at the trailing edge, end at the leading edge


class FixedWing(Design):
    """Twin-motor fixed-wing drone: NACA-section tapered wing with dihedral, streamlined fuselage,
    two wing-mounted nacelles (mirror symmetric) and a flat-plate tail. Wing root leading edge at the
    origin, chord along +X, span along Y, up is +Z. ``part`` selects what is built."""

    parameters = [
        Parameter("part", "aircraft", choices=("aircraft", "wing", "nacelle"), description="what to build"),
        Parameter("span", 1000.0, "mm", min=200),
        Parameter("root_chord", 200.0, "mm", min=50),
        Parameter("taper", 0.7, "", min=0.3, max=1.0, description="tip chord / root chord"),
        Parameter("dihedral_deg", 3.0, "deg", min=0, max=15),
        Parameter("camber", 0.02, "", min=0, max=0.09, description="NACA max camber (2 -> 0.02)"),
        Parameter("camber_pos", 0.4, "", min=0.1, max=0.9, description="NACA camber position (4 -> 0.4)"),
        Parameter("thickness", 0.12, "", min=0.06, max=0.25, description="NACA thickness (12 -> 0.12)"),
        Parameter("fuselage_length", 620.0, "mm", min=100),
        Parameter("fuselage_diameter", 70.0, "mm", min=20),
        Parameter("nose_length", 160.0, "mm", min=20, description="fuselage ahead of the wing leading edge"),
        Parameter("nacelle_y", 300.0, "mm", min=50, description="nacelle centre from the symmetry plane"),
        Parameter("nacelle_diameter", 34.0, "mm", min=10),
        Parameter("nacelle_length", 90.0, "mm", min=20),
        Parameter("nacelle_forward", 45.0, "mm", min=5, description="nacelle ahead of the leading edge"),
        Parameter("tail_span", 360.0, "mm", min=50),
        Parameter("tail_chord", 110.0, "mm", min=20),
        Parameter("fin_height", 130.0, "mm", min=20),
        Parameter("tail_thickness", 5.0, "mm", min=1),
        Parameter("angle_of_attack_deg", 0.0, "deg", min=-10, max=20, description="rotate the aircraft nose-up about Y"),
    ]

    def _sections(self, p):
        up, lo = naca4(p["camber"], p["camber_pos"], p["thickness"])

        def section(wp, c, dx, dz):
            pts_u = [(dx + x * c, dz + y * c) for x, y in up]
            pts_l = [(dx + x * c, dz + y * c) for x, y in lo]
            return (wp.moveTo(*pts_u[0]).spline(pts_u[1:], includeCurrent=True)
                    .spline(pts_l[::-1][1:], includeCurrent=True).close())
        return section

    def _wing(self, p):
        """Constant-chord centre section (inside the fuselage) plus two tapered outer panels with dihedral.
        The centre section has its own faces, so it can be selected as the clamped region in an FEA."""
        c0, c1 = p["root_chord"], p["root_chord"] * p["taper"]
        yc, b2 = p["fuselage_diameter"] / 2 + 5.0, p["span"] / 2
        if b2 <= yc + 10:
            raise ValueError("span too small for the fuselage diameter")
        section = self._sections(p)
        sweep = (c0 - c1) / 4                                  # straight quarter-chord line
        dz = (b2 - yc) * math.tan(math.radians(p["dihedral_deg"]))
        # XZ workplane: normal is -Y, so positive offsets go toward -Y; build the -Y panel and mirror it
        centre = section(cq.Workplane("XZ").workplane(offset=-yc), c0, 0.0, 0.0).extrude(2 * yc)
        panel = (section(cq.Workplane("XZ").workplane(offset=yc), c0, 0.0, 0.0)
                 .workplane(offset=b2 - yc).moveTo(0, 0))
        panel = section(panel, c1, sweep, dz).loft(combine=True, ruled=True)
        return centre.union(panel).union(panel.mirror("XZ"))

    def _nacelle(self, p, y):
        d, L, f = p["nacelle_diameter"], p["nacelle_length"], p["nacelle_forward"]
        body = (cq.Workplane("YZ").workplane(offset=-f).center(y, 0.0).circle(d / 2).extrude(L)
                .faces(">X").edges().fillet(d * 0.3).faces("<X").edges().chamfer(1.5))
        return body

    def build(self, p):
        if p["part"] == "nacelle":
            return self._nacelle(p, 0.0)
        wing = self._wing(p)
        wing = wing.union(self._nacelle(p, -p["nacelle_y"])).union(self._nacelle(p, p["nacelle_y"]))
        if p["part"] == "wing":
            return wing
        # fuselage: elliptic nose, cylinder, tapered tail; axis at z = -D/2 so the wing sits on top
        D, Lf, ln = p["fuselage_diameter"], p["fuselage_length"], p["nose_length"]
        r = D / 2
        lt = 0.45 * Lf
        x0 = -ln
        nose = [(x0 + ln * 0.5 * (1 - math.cos(a)), r * math.sin(a)) for a in (i * math.pi / 2 / 8 for i in range(9))]
        prof = (cq.Workplane("XY").moveTo(x0, 0).spline(nose[1:], includeCurrent=True)
                .lineTo(x0 + Lf - lt, r).lineTo(x0 + Lf, 0.12 * r).lineTo(x0 + Lf, 0).close())
        fuselage = prof.revolve(360, (0, 0, 0), (1, 0, 0)).translate((0, 0, -r))
        # tail: flat plates at the end of the fuselage
        xt = x0 + Lf - p["tail_chord"]
        tt = p["tail_thickness"]
        hstab = cq.Workplane("XY").box(p["tail_chord"], p["tail_span"], tt, centered=(False, True, True)).translate((xt, 0, -r))
        fin = cq.Workplane("XY").box(p["tail_chord"], tt, p["fin_height"], centered=(False, True, False)).translate((xt, 0, -r))
        aircraft = fuselage.union(wing).union(hstab).union(fin)
        if p["angle_of_attack_deg"]:
            aircraft = aircraft.rotate((0, 0, 0), (0, 1, 0), p["angle_of_attack_deg"])    # right-hand rule about +Y: tail down, nose up
        return aircraft
''')
drone = dedalus.load_design(f"{design_file}:FixedWing")
pd.DataFrame(drone.params.table()).set_index("name")

In [ ]:
aircraft = drone.generate()
aircraft          # interactive CadQuery view

In [ ]:
dviz.show(dviz.plot3d(aircraft))

In [ ]:
p0 = drone.resolve()
fig = dviz.plot_sections(aircraft, normal="y", positions=[0.0, p0["nacelle_y"], 0.45 * p0["span"]], cols=3)   # fuselage, nacelle, near tip
fig = dviz.plot_sections(aircraft, normal="x", positions=[-60.0, 60.0, 400.0], cols=3)                          # nose, wing, tail

In [ ]:
b, c0, lam = p0["span"] / 1000, p0["root_chord"] / 1000, p0["taper"]
S = b * c0 * (1 + lam) / 2                       # planform area [m^2] (centre section ~ root chord)
AR = b**2 / S
wing = drone.generate(part="wing"); nacelle = drone.generate(part="nacelle")
airframe_g = (aircraft.surface_area - 2 * nacelle.surface_area) * SHELL_AREAL_DENSITY_G_MM2 + 2 * nacelle.volume * NACELLE_DENSITY_G_MM3
parts.loc["airframe (shells, from CAD area)"] = round(airframe_g, 1)
AUW_kg = parts["mass_g"].sum() / 1000
W = AUW_kg * G
V_CRUISE = 14.0
q = 0.5 * RHO * V_CRUISE**2
print(f"wing area {S:.3f} m^2, aspect ratio {AR:.1f}, airframe {airframe_g:.0f} g, AUW {AUW_kg:.2f} kg")
print(f"wing loading {W / S:.0f} N/m^2, C_L needed at {V_CRUISE} m/s: {W / (q * S):.2f}, "
      f"static thrust-to-weight {2 * THRUST_PER_MOTOR_N / W:.2f}")
parts

## 3. Wing structure (Talos)

The wing + nacelles are analysed as one solid with a **solid-equivalent** modulus for the printed LW-PLA
structure (E = 400 MPa, yield 12 MPa — assumptions to be replaced by coupon tests). Two load cases:

| case | loads | represents |
|---|---|---|
| `pull_up` | lift = 2.5 × W as a uniform pressure on the outer-panel lower skins, + 6 N thrust on each nacelle nose | a 2.5 g pull-up at cruise thrust |
| `engine_out` | lift = 1 × W, 8 N thrust on the **left** nacelle only | full throttle after losing the right motor: asymmetric torsion |

The centre section (|y| < fuselage radius + 5 mm) is clamped — it is glued into the fuselage. The lower
skins are chosen from `talos.inspect_step`: per side, the large B-spline face with the lower centroid.
That choice is printed so you can check it.

In [ ]:
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed; replace with coupon tests")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag
            for right in (False, True)]

def wing_regions(rev):
    p = rev.params
    yc, ny, d, f = p["fuselage_diameter"] / 2 + 5.0, p["nacelle_y"], p["nacelle_diameter"] / 2 + 1, p["nacelle_forward"]
    return [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)),
            talos.Surfaces("lift", lower_skins(rev.step)),
            talos.SurfacesInBox("motor_left", (-f - 0.1, -ny - d, -d, -f + 0.1, -ny + d, d)),
            talos.SurfacesInBox("motor_right", (-f - 0.1, ny - d, -d, -f + 0.1, ny + d, d))]

def wing_model(rev, loads, name):
    return talos.StructuralModel(rev.step, "mm-N-MPa", LW_PLA, wing_regions(rev), [talos.FixedSupport("root")], loads,
                                 talos.MeshSettings(element_size=10.0), name=name)

def lift_pressure_MPa(n):                       # n x weight spread over the planform, in MPa (N/mm^2)
    return n * W / (S * 1e6)

def pull_up(rev):
    return wing_model(rev, [talos.Pressure("lift", lift_pressure_MPa(2.5)),
                            talos.Force("motor_left", fx=6.0), talos.Force("motor_right", fx=6.0)], "pull_up")

def engine_out(rev):
    return wing_model(rev, [talos.Pressure("lift", lift_pressure_MPa(1.0)), talos.Force("motor_left", fx=8.0)], "engine_out")

print(f"lift pressure at 2.5 g: {lift_pressure_MPa(2.5) * 1e6:.0f} Pa")

In [ ]:
ws = core.Workspace.create(RUNS / "workspace", name="fixed-wing drone")
design = ws.add_design("drone", f"{design_file}:FixedWing")
w1 = design.new_revision(part="wing", note="baseline wing NACA 2412, taper 0.7")
w1.generate()
print("lower skins:", lower_skins(w1.step))
talos.inspect_step(w1.step, units="mm-N-MPa")

In [ ]:
ev_pu = w1.run_fea("pull_up", pull_up, progress=True)
ev_eo = w1.run_fea("engine_out", engine_out, progress=True)
ws.status()

In [ ]:
tviz.show(tviz.plot_problem(pull_up(w1), ev_pu.tool_results[0], arrow_scale=60))

In [ ]:
if ev_pu.ok:
    tviz.show(tviz.plot_results(ev_pu.tool_results[-1], field="von_mises"))

In [ ]:
if ev_pu.ok:
    res = ev_pu.tool_results[-1]
    fig = tviz.plot_section(res, normal="x", origin=(60.0, 0, 0), field="von_mises")      # spanwise: bending stress
    fig = tviz.plot_section(res, normal="y", origin=(0, 250.0, 0), field="von_mises")     # airfoil section at mid-panel
    fig = talos.plot_along_axis(res, axis="y", quantity="displacement", component=2)     # spanwise deflection

In [ ]:
if ev_eo.ok:
    tviz.show(tviz.plot_results(ev_eo.tool_results[-1], field="|U|"))
    print(ev_eo)

### Reading the numbers
`max_displacement` is the tip deflection; `safety_factor_yield` is yield / peak nodal von Mises, which
sits at the clamp or at the nacelle-wing junction (a stress concentration; compare between revisions,
do not read it as an absolute). In `engine_out` look at the *asymmetric* displacement: the thrust twists
the left panel.

## 4. A thicker wing? Two revisions compared

A NACA 2415 (`thickness=0.15`) is stiffer (bending stiffness ~ t³) at a small drag cost. Same load
cases, same factories, the workspace keeps both.

In [ ]:
w2 = next((r for r in ws.revisions() if r.record.get("note") == "NACA 2415: thicker wing"), None) \
     or w1.branch(thickness=0.15, note="NACA 2415: thicker wing")
if not w2.is_generated:
    w2.generate()
for case in tqdm((pull_up, engine_out), desc="load cases"):          # safe to interrupt and re-run
    if w2.evaluation("fea", case.__name__) is None:
        w2.run_fea(case.__name__, case, progress=True)

def row(rev):
    vol_mm3 = rev.geometry_summary()["metrics"]["volume"]
    d = {"rev": rev.id, "note": rev.record.get("note", ""), "thickness": rev.params["thickness"],
         "wing_volume_cm3": vol_mm3 / 1e3}
    for case in ("pull_up", "engine_out"):
        ev = rev.evaluation("fea", case)
        d[f"{case}_tip_mm"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case}_SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

table = pd.DataFrame([row(r) for r in (w1, w2)]).set_index("rev").round(2)
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
table.plot.bar(y=["pull_up_tip_mm", "engine_out_tip_mm"], ax=ax[0], title="tip deflection [mm]", rot=0)
table.plot.bar(y=["pull_up_SF", "engine_out_SF"], ax=ax[1], title="safety factor (yield)", rot=0)
fig.tight_layout()
best = table["pull_up_SF"].idxmax()
preferred_wing = ws.revision(best)
preferred_wing.label("preferred", note=f"stiffer; pull-up SF {table.loc[best, 'pull_up_SF']}")
ws.status()

## 5. Whole-aircraft aerodynamics at 4° (Aeromant)

Steady RANS (k-ω SST) of the complete aircraft with the preferred wing thickness, 14 m/s, 4° angle of
attack, coarse mesh (a few minutes). The template's domain scales with `reference_length`, which
must be ≥ span/4 here (`max_body_extent`), so **Cm is referenced to 0.25 m**; Cl and Cd use the wing
area. The engineering outputs: lift and drag at this attitude, the level-flight speed at which this
Cl carries the aircraft, and cruise power → endurance.

In [ ]:
a1 = preferred_wing.branch(part="aircraft", angle_of_attack_deg=4.0, note="whole aircraft, 4 deg AoA, for CFD")
a1.generate(stl_tolerance=0.2)
cases = {}

def cruise_4deg(rev, workdir):
    case = aeromant.CFDCase(
        "rans_ksst_external", rev.stl,
        dict(velocity=V_CRUISE, kinematic_viscosity=1.5e-5, density=RHO, reference_area=S, reference_length=0.25,
             center_of_rotation=(0.05, 0.0, 0.0), iterations=400, residual_target=1e-4,
             surface_level=4, near_level=3, wake_level=2, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    cases[rev.id] = case
    return case

ev_cfd = a1.run_cfd("cruise_4deg", cruise_4deg, progress=True)
print(ev_cfd)

In [ ]:
case = cases[a1.id]
aviz.show(aviz.plot_setup(case))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_mesh_slice(case, normal="y", origin=(0, 0.0, 0)))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="y", origin=(0, p0["nacelle_y"] / 1000, 0)))   # through a nacelle
    fig = aviz.plot_section(case, "p", normal="y", origin=(0, 0.0, 0), zoom=2)                       # fuselage plane
    fig = aviz.plot_section(case, "U", normal="x", origin=(0.10, 0, 0), zoom=1.5)                    # behind the leading edge: wing + nacelles

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_streamlines(case, n=80))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_surface_pressure(case))
    fig = aeromant.plot_coefficients(case.workdir)

In [ ]:
if ev_cfd.ok:
    m = ev_cfd.metrics
    Cl, Cd = m["Cl"], m["Cd"]
    V_level = math.sqrt(2 * W / (RHO * S * Cl)) if Cl > 0 else float("nan")   # speed at which this Cl carries W
    D_level = 0.5 * RHO * V_level**2 * S * Cd
    P_cruise = D_level * V_level / PROP_EFFICIENCY                              # electrical-ish power at the props
    endurance_min = BATTERY_WH * USABLE_FRACTION / P_cruise * 60 if P_cruise > 0 else float("nan")
    aero = pd.Series({
        "Cl (4 deg)": Cl, "Cd (4 deg)": Cd, "L/D": Cl / Cd,
        "lift at 14 m/s [N]": m["lift_force_N"], "weight [N]": W,
        "level-flight speed at this Cl [m/s]": V_level, "drag there [N]": D_level,
        "cruise power (prop eff. 0.55) [W]": P_cruise, "endurance (80 % of battery) [min]": endurance_min,
        "converged": m["converged"], "cells": m["mesh_cells"],
    })
    print(aero.to_string())

The mesh is coarse and the run short: treat these as a **first estimate** (Cl at 4° should land in the
0.4–0.6 range for this wing; the trailing edge is not resolved). The angle of attack is a design
parameter, so a sweep is three more branches (`angle_of_attack_deg=0, 8, 12`) and one table — a job for
the comparison tooling of Stage 11, or for a loop you write here.

## 6. Print a nacelle (Mellonia)

Nose down on the bed: the chamfered motor face is flat, so `rotate_y=90` puts it on the plate. Two
copies are needed — slice once, print twice.

In [ ]:
n1 = design.new_revision(part="nacelle", note="nacelle for printing")
n1.generate()
ev_p = n1.run_print("nose_down", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90))
print(ev_p)

In [ ]:
if ev_p.ok:
    prn = ev_p.tool_results[0]
    mviz.show(mviz.plot_toolpath(prn))
    fig = mviz.plot_layer_grid(prn, n=6, cols=3)

## 7. Where we are

One design file, four revisions, six evaluations — every number above can be traced to a folder with
the tool's native files and the exact commands that produced it.

In [ ]:
ws.status()

In [ ]:
for p in sorted((RUNS / "workspace" / "revisions" / a1.id).rglob("*"))[:20]:
    print(p.relative_to(RUNS / "workspace"))

**Next steps an engineer would take:** an angle-of-attack sweep and the tail sizing from Cm; a carbon
spar (a second solid in the wing, bonded); a proper printed-shell wing model (Talos is a solid-element
tool — a shell analysis is a different model); replacing the assumed LW-PLA properties with coupon
tests; and letting the AI copilot (notebook 07) propose weight savings on `_runs/fixed_wing/wing.py`,
each proposal built and measured before you accept it.